# Module 06 — Notebook 3: Eval Script Mini-Project

## What You'll Build

`scripts/run_evaluation.py` — a command-line evaluation tool that:

1. Accepts `--input` (JSON file), `--output` (results JSON), and `--threshold` (float)
2. Loads model output data
3. Computes flag rate per model and overall
4. Warns if any model exceeds the threshold
5. Saves a summary JSON to `--output`
6. Logs all progress with the `logging` module

You'll build it incrementally — one function at a time — and test each piece before assembling the whole.

**Time:** ~25 minutes

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_contains, check_keys
import json
import logging
import argparse
from pathlib import Path
import subprocess

SCRIPTS_DIR = Path("scripts")
SCRIPTS_DIR.mkdir(exist_ok=True)

DATA_JSON  = Path("../../data/synthetic/model_outputs.json")
DATA_CSV   = Path("../../data/synthetic/evaluation_results.csv")
OUTPUT_DIR = Path("../../output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("data files:", DATA_JSON.exists(), DATA_CSV.exists())

## Step 1: The Core Functions

We'll write each function in the notebook first to test it, then assemble them into the script.

Start with loading and summarizing data — pure functions, easy to test.

In [ ]:
def load_outputs(path):
    """Load model outputs from a JSON file."""
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def compute_summary(outputs, threshold=0.5):
    """Compute overall and per-model flag statistics."""
    total    = len(outputs)
    flagged  = sum(1 for o in outputs if o["flagged"])
    overall  = flagged / total if total else 0.0

    # Per-model breakdown
    from collections import defaultdict
    counts = defaultdict(lambda: {"total": 0, "flagged": 0})
    for o in outputs:
        counts[o["model"]]["total"]   += 1
        counts[o["model"]]["flagged"] += int(o["flagged"])

    models_summary = {}
    alerts = []
    for model, c in sorted(counts.items()):
        rate = c["flagged"] / c["total"]
        models_summary[model] = {
            "total":     c["total"],
            "flagged":   c["flagged"],
            "flag_rate": round(rate, 4),
        }
        if rate > threshold:
            alerts.append(f"{model}: {rate:.1%} (threshold={threshold:.0%})")

    return {
        "total_outputs":    total,
        "total_flagged":    flagged,
        "overall_flag_rate": round(overall, 4),
        "models":           models_summary,
        "alerts":           alerts,
    }


# Quick test
outputs = load_outputs(DATA_JSON)
summary = compute_summary(outputs, threshold=0.5)
print(json.dumps(summary, indent=2))

## Step 2: Saving Results

A script that only prints is hard to use in a pipeline — the next step can't read it.
Always save structured output to a file.

In [ ]:
def save_results(summary, output_path):
    """Write the summary dict to a JSON file, creating parent dirs as needed."""
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)


# Test it
test_output = OUTPUT_DIR / "test_summary.json"
save_results(summary, test_output)

print(f"Saved to {test_output}")
print("File exists:", test_output.exists())

# Verify round-trip
with open(test_output) as f:
    loaded = json.load(f)
print("Round-trip check:", loaded["total_outputs"] == summary["total_outputs"])

## Step 3: Assemble the Full Script

Now we put it all together — argparse, logging, the helper functions, and the entry point.

In [ ]:
%%writefile scripts/run_evaluation.py
"""run_evaluation.py — compute flag statistics from model output JSON.

Usage:
    python run_evaluation.py --input data.json --output results.json
    python run_evaluation.py --input data.json --threshold 0.4 --verbose
"""
import argparse
import json
import logging
from collections import defaultdict
from pathlib import Path


def build_parser():
    parser = argparse.ArgumentParser(description="Evaluate model output flag rates.")
    parser.add_argument("--input",     type=Path, required=True,
                        help="Path to model_outputs.json")
    parser.add_argument("--output",    type=Path, default=Path("output/evaluation_results.json"),
                        help="Where to save the summary JSON")
    parser.add_argument("--threshold", type=float, default=0.5,
                        help="Flag rate above this triggers a warning (default 0.5)")
    parser.add_argument("--verbose",   action="store_true",
                        help="Enable DEBUG logging")
    return parser


def load_outputs(path):
    logger = logging.getLogger(__name__)
    logger.debug(f"Opening {path}")
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    logger.info(f"Loaded {len(data)} outputs from {path}")
    return data


def compute_summary(outputs, threshold):
    total   = len(outputs)
    flagged = sum(1 for o in outputs if o["flagged"])
    overall = flagged / total if total else 0.0

    counts = defaultdict(lambda: {"total": 0, "flagged": 0})
    for o in outputs:
        counts[o["model"]]["total"]   += 1
        counts[o["model"]]["flagged"] += int(o["flagged"])

    logger = logging.getLogger(__name__)
    models_summary = {}
    alerts = []
    for model, c in sorted(counts.items()):
        rate = c["flagged"] / c["total"]
        models_summary[model] = {
            "total":     c["total"],
            "flagged":   c["flagged"],
            "flag_rate": round(rate, 4),
        }
        logger.debug(f"{model}: {c['flagged']}/{c['total']} flagged ({rate:.1%})")
        if rate > threshold:
            alert = f"{model}: {rate:.1%} exceeds threshold {threshold:.0%}"
            logger.warning(alert)
            alerts.append(alert)

    return {
        "total_outputs":     total,
        "total_flagged":     flagged,
        "overall_flag_rate": round(overall, 4),
        "threshold":         threshold,
        "models":            models_summary,
        "alerts":            alerts,
    }


def save_results(summary, output_path):
    logger = logging.getLogger(__name__)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    logger.info(f"Saved results to {output_path}")


def main(args):
    level = logging.DEBUG if args.verbose else logging.INFO
    logging.basicConfig(level=level, format="%(levelname)-8s %(message)s")
    logger = logging.getLogger(__name__)

    logger.info("Starting evaluation")
    outputs = load_outputs(args.input)
    summary = compute_summary(outputs, threshold=args.threshold)

    logger.info(
        f"Overall flag rate: {summary['overall_flag_rate']:.1%} "
        f"({summary['total_flagged']}/{summary['total_outputs']} flagged)"
    )

    if summary["alerts"]:
        logger.warning(f"{len(summary['alerts'])} model(s) exceed threshold")
    else:
        logger.info("All models are within threshold")

    save_results(summary, args.output)
    logger.info("Done")
    return summary


if __name__ == "__main__":
    main(build_parser().parse_args())

In [ ]:
# Run it — should log INFO messages and one WARNING for model-b-v1
!python scripts/run_evaluation.py \
    --input ../../data/synthetic/model_outputs.json \
    --output ../../output/eval_summary.json \
    --threshold 0.5

In [ ]:
# Inspect the saved JSON
with open("../../output/eval_summary.json") as f:
    result = json.load(f)

print(json.dumps(result, indent=2))

---
## Your Turn — Exercise 1: Test the Core Functions

Import `run_evaluation` and test its helper functions directly.

1. Load outputs using `run_evaluation.load_outputs(DATA_JSON)` and store in `outputs`.
2. Call `run_evaluation.compute_summary(outputs, threshold=0.5)` and store in `summary`.
3. Store `summary["total_flagged"]` in `n_flagged`.
4. Store the flag rate for `"model-b-v1"` in `b1_rate` (rounded to 4 decimal places).

In [ ]:
# Add scripts/ to path so we can import run_evaluation
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import run_evaluation
import importlib
importlib.reload(run_evaluation)

# YOUR CODE HERE
outputs  = None   # run_evaluation.load_outputs(DATA_JSON)
summary  = None   # run_evaluation.compute_summary(outputs, threshold=0.5)
n_flagged = None  # integer from summary
b1_rate   = None  # float, rounded to 4 decimal places

In [ ]:
check_type(outputs, list, "outputs is a list")
check_equal(len(outputs), 20, "20 outputs loaded")
check_type(summary, dict, "summary is a dict")
check_equal(int(n_flagged), 7, "7 outputs flagged")
check_approx(b1_rate, 0.7778, 1e-3, "model-b-v1 flag rate")

---
## Your Turn — Exercise 2: Run the Script and Load the Output

Run `run_evaluation.py` using `subprocess.run()` — the programmatic way to call a script
from Python (rather than `!` which only works in notebooks).

1. Run the script against `DATA_JSON` with `threshold=0.3` and output to `../../output/ex2_results.json`.
2. Store `result.returncode` in `exit_code` (should be `0` for success).
3. Load the output JSON into `ex2_summary`.
4. Store `ex2_summary["alerts"]` in `alerts` — at threshold 0.3, both model-b versions should be flagged.

In [ ]:
# YOUR CODE HERE
result = None   # subprocess.run(["python", "scripts/run_evaluation.py", ...], capture_output=True, text=True)
exit_code   = None   # integer
ex2_summary = None   # dict loaded from the output JSON
alerts      = None   # list from ex2_summary

In [ ]:
check_equal(int(exit_code), 0, "script exited with code 0 (success)")
check_type(ex2_summary, dict, "ex2_summary is a dict")
check_type(alerts, list, "alerts is a list")
# At threshold 0.3: model-b-v1 (0.78) and model-b-v2 (0) ... let's check b-v1 at least
# model-b-v1 has rate 0.7778, model-b-v2 has rate 0.0 — only b-v1 fires
check_equal(len(alerts) >= 1, True, "at least 1 alert at threshold 0.3")

---
## Your Turn — Exercise 3: Extend the Script

Add a `--model` argument to `run_evaluation.py` that filters outputs to a single model before computing stats.

1. Use `%%writefile` to create `scripts/run_evaluation_v2.py` with the `--model` argument added.
   - Add: `parser.add_argument("--model", type=str, default=None, help="Filter to one model")`
   - In `main()`: if `args.model` is set, filter `outputs` to only entries where `output["model"] == args.model`
   - Log: `f"Filtered to {len(outputs)} outputs for model '{args.model}'"`
2. Run it with `--model model-b-v1` and store the overall flag rate in `b1_only_rate`.

> For model-b-v1 (9 outputs, 7 flagged): expected rate = 7/9 ≈ 0.7778

In [ ]:
%%writefile scripts/run_evaluation_v2.py
# YOUR CODE HERE — copy run_evaluation.py and add --model filtering
print("replace me")

In [ ]:
# Run with --model model-b-v1
ex3_result = subprocess.run(
    ["python", "scripts/run_evaluation_v2.py",
     "--input", str(DATA_JSON),
     "--output", "../../output/ex3_results.json",
     "--model", "model-b-v1"],
    capture_output=True, text=True
)
print(ex3_result.stdout)
print(ex3_result.stderr)

In [ ]:
# YOUR CODE HERE: load the output JSON and extract overall_flag_rate
b1_only_rate = None   # float: overall_flag_rate for model-b-v1 only

In [ ]:
check_equal(ex3_result.returncode, 0, "v2 script exited cleanly")
check_approx(b1_only_rate, 0.7778, 1e-3, "model-b-v1 only flag rate is 0.7778")

---
## Why This Matters for AI Research Engineering

`run_evaluation.py` is a realistic eval pipeline script — the kind of tool you'd write to:
- Run nightly on new model outputs from a training run
- Compare models side-by-side before a deployment decision
- Integrate into a CI/CD pipeline that gates deployments on safety metrics

The three properties that make it production-worthy:

1. **CLI interface via argparse** — schedulers, CI systems, and teammates can call it without touching the code
2. **Structured JSON output** — the next step in the pipeline (a report generator, a dashboard, a Slack notification) can read it reliably
3. **Logging with levels** — `WARNING` fires for safety-relevant events; `INFO` tracks progress; both are queryable in a log aggregation system

The `subprocess.run()` pattern from Exercise 2 is how you test scripts in automated testing — you run the script as a black box and check its output file and exit code. That's the same way CI systems test scripts.

## Summary — Module 06 Complete

| Topic | Key patterns |
|-------|--------------|
| Script structure | imports → constants → functions → `main()` → entry point |
| Entry point guard | `if __name__ == "__main__": main(...)` |
| Write files from notebook | `%%writefile scripts/my_script.py` |
| Run scripts | `!python scripts/my_script.py` (notebook) or `subprocess.run([...])` (Python) |
| argparse | `ArgumentParser` + `add_argument` + `parse_args()` |
| Typed args | `type=float`, `type=Path`, `action="store_true"` |
| Logging | `basicConfig(level=INFO)` + `getLogger(__name__)` |
| Log levels | DEBUG (verbose) → INFO (progress) → WARNING (safety) → ERROR (failures) |

**Next:** Module 07 — APIs and HTTP requests, building on these script-writing skills to call external services.